In [4]:

import pygame
import sys
import math

pygame.init()

# Window settings
WIDTH = 600
HEIGHT = 700
BOARD_SIZE = 600
CELL_SIZE = BOARD_SIZE // 3

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Tic Tac Toe - Minimax AI")

# Colors
WHITE = (255, 255, 255)
BLACK = (20, 20, 20)
BLUE = (40, 110, 220)
RED = (220, 65, 65)
GREEN = (30, 160, 90)
GRAY = (180, 180, 180)

# Fonts
font = pygame.font.SysFont(None, 110)
status_font = pygame.font.SysFont(None, 38)
small_font = pygame.font.SysFont(None, 28)

# Game variables
board = []
game_over = False
status = "Your Turn (X)"

HUMAN = "X"
AI = "O"


# Restart game
def restart_game():
    global board, game_over, status

    board = [
        ["", "", ""],
        ["", "", ""],
        ["", "", ""]
    ]

    game_over = False
    status = "Your Turn (X)"


# Check winner
def check_winner():

    for row in board:
        if row[0] == row[1] == row[2] != "":
            return row[0]

    for col in range(3):
        if board[0][col] == board[1][col] == board[2][col] != "":
            return board[0][col]

    if board[0][0] == board[1][1] == board[2][2] != "":
        return board[0][0]

    if board[0][2] == board[1][1] == board[2][0] != "":
        return board[0][2]

    return None


# Check draw
def is_draw():

    for row in board:
        for cell in row:
            if cell == "":
                return False

    return True


# Minimax Algorithm
def minimax(is_ai_turn):

    winner = check_winner()

    # AI wins
    if winner == AI:
        return 1

    # Human wins
    if winner == HUMAN:
        return -1

    # Draw
    if is_draw():
        return 0

    # AI = MAX player
    if is_ai_turn:

        best_score = -math.inf

        for row in range(3):
            for col in range(3):

                if board[row][col] == "":

                    board[row][col] = AI

                    score = minimax(False)

                    # Undo move
                    board[row][col] = ""

                    best_score = max(best_score, score)

        return best_score

    # Human = MIN player
    else:

        best_score = math.inf

        for row in range(3):
            for col in range(3):

                if board[row][col] == "":

                    board[row][col] = HUMAN

                    score = minimax(True)

                    # Undo move
                    board[row][col] = ""

                    best_score = min(best_score, score)

        return best_score


# AI move using Minimax
def ai_move():

    best_score = -math.inf
    best_move = None

    for row in range(3):
        for col in range(3):

            if board[row][col] == "":

                board[row][col] = AI

                score = minimax(False)

                # Undo temporary move
                board[row][col] = ""

                if score > best_score:
                    best_score = score
                    best_move = (row, col)

    if best_move is not None:

        row, col = best_move
        board[row][col] = AI


# Draw board
def draw_board():

    screen.fill(WHITE)

    # Grid lines
    for i in range(1, 3):

        pygame.draw.line(
            screen,
            BLACK,
            (0, i * CELL_SIZE),
            (BOARD_SIZE, i * CELL_SIZE),
            5
        )

        pygame.draw.line(
            screen,
            BLACK,
            (i * CELL_SIZE, 0),
            (i * CELL_SIZE, BOARD_SIZE),
            5
        )

    # Draw X and O
    for row in range(3):
        for col in range(3):

            x = col * CELL_SIZE + CELL_SIZE // 2
            y = row * CELL_SIZE + CELL_SIZE // 2

            if board[row][col] == HUMAN:

                text = font.render("X", True, BLUE)
                rect = text.get_rect(center=(x, y))
                screen.blit(text, rect)

            elif board[row][col] == AI:

                text = font.render("O", True, RED)
                rect = text.get_rect(center=(x, y))
                screen.blit(text, rect)


# Draw status
def draw_status():

    pygame.draw.rect(
        screen,
        WHITE,
        (0, BOARD_SIZE, WIDTH, HEIGHT - BOARD_SIZE)
    )

    text = status_font.render(status, True, BLACK)
    rect = text.get_rect(center=(WIDTH // 2, 630))
    screen.blit(text, rect)

    restart_text = small_font.render(
        "Press R to Restart | ESC to Exit",
        True,
        GREEN
    )

    restart_rect = restart_text.get_rect(
        center=(WIDTH // 2, 675)
    )

    screen.blit(restart_text, restart_rect)


# Initial game
restart_game()

running = True

# Main game loop
while running:

    for event in pygame.event.get():

        if event.type == pygame.QUIT:
            running = False

        # Keyboard
        if event.type == pygame.KEYDOWN:

            if event.key == pygame.K_ESCAPE:
                running = False

            elif event.key == pygame.K_r:
                restart_game()

        # Mouse click
        if event.type == pygame.MOUSEBUTTONDOWN:

            if event.button == 1 and not game_over:

                mouse_x, mouse_y = event.pos

                if mouse_y < BOARD_SIZE:

                    row = mouse_y // CELL_SIZE
                    col = mouse_x // CELL_SIZE

                    # Human move
                    if board[row][col] == "":

                        board[row][col] = HUMAN

                        winner = check_winner()

                        if winner == HUMAN:
                            status = "You Win!"
                            game_over = True

                        elif is_draw():
                            status = "It's a Draw!"
                            game_over = True

                        else:

                            status = "AI is Thinking..."

                            draw_board()
                            draw_status()
                            pygame.display.flip()

                            pygame.time.delay(250)

                            # AI uses Minimax
                            ai_move()

                            winner = check_winner()

                            if winner == AI:
                                status = "AI Wins!"
                                game_over = True

                            elif is_draw():
                                status = "It's a Draw!"
                                game_over = True

                            else:
                                status = "Your Turn (X)"

    # Update screen
    draw_board()
    draw_status()

    pygame.display.flip()


pygame.quit()
sys.exit()


SystemExit: 